# EstetiScan AI — Demonstração Técnica

**Sistema Automatizado de Triagem, Padronização e Segmentação de Imagens para Clínicas de Estética**

---

### O que este notebook demonstra

Este notebook apresenta o motor técnico do **EstetiScan AI**, uma solução capaz de:

1. **Analisar automaticamente** a qualidade de imagens clínicas (brilho, contraste, nitidez);
2. **Corrigir imagens recuperáveis** via CLAHE no canal de luminância;
3. **Validar o ganho real** da correção (sem aceitar ajustes que piorem a imagem);
4. **Verificar a viabilidade de segmentação** pelo critério de Otsu (separabilidade inter-classe);
5. **Segmentar** a região de interesse quando houver confiança suficiente;
6. **Classificar** cada imagem em um dos cinco status:
   - `ideal` — pronta, sem necessidade de correção
   - `adequada` — utilizável mas sem segmentação viável
   - `corrigida_automaticamente` — recuperada com sucesso
   - `ajuste_sem_ganho_relevante` — correção tentada sem melhoria real
   - `requer_revisao_humana` — complexa demais para processamento automático

> **Nota ética:** o sistema não substitui avaliação profissional. Atua como ferramenta de triagem, padronização e apoio visual.

---
## 1. Instalação e Setup do Ambiente

Configuração para execução no **Google Colab**. Se estiver rodando localmente, pule esta célula.

In [ ]:
# ==== Setup para Google Colab ====
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content/estetiscan")

    if not repo_dir.exists():
        !git clone https://github.com/galsz/estetiscan.git /content/estetiscan
    else:
        os.chdir(repo_dir)
        !git fetch origin
        !git checkout main
        !git pull --ff-only origin main

    os.chdir(repo_dir)
    PROJECT_ROOT = repo_dir.resolve()
else:
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir

requirements_path = PROJECT_ROOT / "requirements.txt"

if requirements_path.exists():
    !pip install -q -r "{requirements_path}"
else:
    print(f"requirements.txt não encontrado em: {requirements_path}")
    print("Arquivos encontrados na raiz do projeto:")
    for item in sorted(PROJECT_ROOT.iterdir()):
        print(f" - {item.name}")
    print("Instalando dependências mínimas para a demo...")
    !pip install -q opencv-python numpy pandas matplotlib

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Diretório do projeto: {PROJECT_ROOT}")
print(f"requirements.txt: {requirements_path}")
print(f"Google Colab: {IN_COLAB}")

---
## 2. Importação das Bibliotecas e do Motor

Importamos as bibliotecas de visualização e as funções do motor pela **fachada em PT-BR** (`facade.py`).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, HTML

# Funções do motor (fachada PT-BR)
from motor.facade import (
    analisar_qualidade,
    aplicar_equalizacao,
    verificar_viabilidade_segmentacao,
    segmentar_imagem,
    processar_imagem,
    processar_lote,
    gerar_relatorio,
    imprimir_resumo,
)
from motor.io import read_color, to_grayscale, list_images

print("✔ Motor EstetiScan AI carregado com sucesso.")

---
## 3. Funções Auxiliares de Visualização

Funções utilitárias para exibir resultados de forma padronizada no notebook.

In [ ]:
# Cores para os status
_STATUS_COLORS = {
    "ideal": "#2ecc71",
    "adequada": "#3498db",
    "corrigida_automaticamente": "#f39c12",
    "ajuste_sem_ganho_relevante": "#e67e22",
    "requer_revisao_humana": "#e74c3c",
}


def exibir_resultado(resultado: dict) -> None:
    """Exibe o painel visual completo de uma imagem processada."""
    artefatos = resultado.get("artefatos", {})
    status = resultado["status_cliente"]
    cor = _STATUS_COLORS.get(status, "#95a5a6")

    # Carregar imagens dos artefatos salvos
    img_orig = cv2.imread(artefatos["orig_color"]) if artefatos.get("orig_color") else None
    img_gray = cv2.imread(artefatos["gray"], cv2.IMREAD_GRAYSCALE) if artefatos.get("gray") else None
    img_eq = cv2.imread(artefatos["eq_color"]) if artefatos.get("eq_color") else None
    img_seg = cv2.imread(artefatos["seg_color"]) if artefatos.get("seg_color") else None

    # Determinar quantos painéis mostrar
    panels = ["Original", "Histograma"]
    if img_eq is not None:
        panels.append("Corrigida (CLAHE)")
    if img_seg is not None:
        panels.append("Segmentada")

    n_panels = len(panels)
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4.5))
    if n_panels == 1:
        axes = [axes]

    idx = 0

    # --- Original ---
    if img_orig is not None:
        axes[idx].imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
        axes[idx].set_title("Original")
        axes[idx].axis("off")
    idx += 1

    # --- Histograma ---
    if img_gray is not None:
        axes[idx].hist(img_gray.ravel(), bins=256, range=(0, 256),
                       color="gray", alpha=0.8)
        axes[idx].set_title("Histograma")
        axes[idx].set_xlabel("Intensidade")
        axes[idx].set_ylabel("Frequência")

        # Marcar limiar de Otsu se disponível
        limiar = resultado.get("limiar_otsu")
        if limiar is not None:
            axes[idx].axvline(limiar, color="red", linestyle="--",
                              label=f"Otsu = {limiar}")
            axes[idx].legend(fontsize=8)
    idx += 1

    # --- Corrigida ---
    if img_eq is not None:
        axes[idx].imshow(cv2.cvtColor(img_eq, cv2.COLOR_BGR2RGB))
        axes[idx].set_title("Corrigida (CLAHE)")
        axes[idx].axis("off")
        idx += 1

    # --- Segmentada ---
    if img_seg is not None:
        axes[idx].imshow(cv2.cvtColor(img_seg, cv2.COLOR_BGR2RGB))
        axes[idx].set_title("Segmentada")
        axes[idx].axis("off")

    # Título geral com status
    metricas_orig = resultado.get("metricas_originais", {})
    sep = resultado.get("separabilidade_otsu")
    sep_str = f"{sep:.2f}" if sep is not None else "N/A"

    fig.suptitle(
        f"{resultado['arquivo']}\n"
        f"Status: {status.upper()}  |  "
        f"Média: {metricas_orig.get('mean', 0):.1f}  |  "
        f"Std: {metricas_orig.get('std', 0):.1f}  |  "
        f"Separabilidade: {sep_str}",
        fontsize=11,
        fontweight="bold",
        color=cor,
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

---
## 4. Upload e Processamento de Uma Única Imagem

Aqui demonstramos o pipeline completo para **uma imagem individual**.

- No **Colab**: usa `google.colab.files.upload()` para upload interativo.
- **Localmente**: aponta para um arquivo do dataset.

In [ ]:
# Upload de uma única imagem
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    caminho_imagem = Path(list(uploaded.keys())[0])
else:
    # Localmente: usa a primeira imagem do dataset como exemplo
    imagens_disponiveis = list_images(Path(PROJECT_ROOT) / "data" / "original")
    caminho_imagem = imagens_disponiveis[0]

print(f"Imagem selecionada: {caminho_imagem.name}")

# Pré-visualização
img_preview = cv2.imread(str(caminho_imagem))
plt.figure(figsize=(6, 5))
plt.imshow(cv2.cvtColor(img_preview, cv2.COLOR_BGR2RGB))
plt.title(f"Upload: {caminho_imagem.name}")
plt.axis("off")
plt.show()

### 4.1 Processamento pelo pipeline completo

A função `processar_imagem()` executa as 5 etapas automaticamente:

1. Leitura e conversão para escala de cinza
2. Análise de qualidade (média, desvio padrão, Laplaciano, clipping)
3. Correção automática via CLAHE + validação de ganho
4. Verificação de viabilidade de segmentação (Otsu)
5. Segmentação (se viável) e classificação final

In [ ]:
# Processar a imagem pelo pipeline completo
resultado_unico = processar_imagem(caminho_imagem)

# Exibir resultado estruturado (resumo textual)
print(f"Arquivo:        {resultado_unico['arquivo']}")
print(f"Status:         {resultado_unico['status_cliente']}")
print(f"Equalização:    {'Sim' if resultado_unico['needs_equalization'] else 'Não'}")
print(f"Revisão humana: {'Sim' if resultado_unico['requires_human_review'] else 'Não'}")
print(f"Separabilidade: {resultado_unico['separabilidade_otsu']:.4f}" if resultado_unico['separabilidade_otsu'] else "")
print(f"Limiar Otsu:    {resultado_unico['limiar_otsu']}" if resultado_unico['limiar_otsu'] else "")

### 4.2 Visualização detalhada

Painel visual com: imagem original, histograma (com limiar de Otsu), imagem corrigida (se houve correção) e imagem segmentada (se viável).

In [ ]:
exibir_resultado(resultado_unico)

---
## 5. Upload de Múltiplas Imagens

No **Colab**, permite upload de vários arquivos de uma vez.
**Localmente**, usa o diretório `data/original/` como fonte.

In [ ]:
# Diretório com as imagens
if IN_COLAB:
    from google.colab import files
    print("Faça upload das imagens (selecione múltiplos arquivos):")
    uploaded_files = files.upload()
    # Salvar em pasta temporária
    pasta_upload = Path("upload_temp")
    pasta_upload.mkdir(exist_ok=True)
    for nome, conteudo in uploaded_files.items():
        (pasta_upload / nome).write_bytes(conteudo)
    pasta_imagens = pasta_upload
else:
    pasta_imagens = Path(PROJECT_ROOT) / "data" / "original"

# Listar imagens encontradas
imagens = list_images(pasta_imagens)
print(f"\n{len(imagens)} imagens encontradas:")
for img in imagens:
    print(f"  • {img.name}")

In [ ]:
# Grid de thumbnails para confirmar as imagens carregadas
n_imgs = len(imagens)
cols = min(6, n_imgs)
rows = (n_imgs + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = np.array(axes).flatten() if n_imgs > 1 else [axes]

for i, img_path in enumerate(imagens):
    img = cv2.imread(str(img_path))
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(img_path.stem[:20] + "...", fontsize=7)
    axes[i].axis("off")

# Ocultar eixos vazios
for j in range(n_imgs, len(axes)):
    axes[j].axis("off")

fig.suptitle(f"Dataset: {n_imgs} imagens", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6. Processamento em Lote

A função `processar_lote()` processa **todas as imagens do diretório** automaticamente, sem ajuste manual por imagem.

Cada imagem passa pelas 5 etapas do pipeline e recebe uma classificação final.

In [ ]:
# Processamento em lote
batch = processar_lote(pasta_imagens)

resultados = batch["resultados"]
resumo = batch["resumo"]

# Exibir resumo no console
imprimir_resumo(resumo)

---
## 7. Visualização dos Resultados por Imagem

Painel visual completo para cada imagem processada: original, histograma, correção (se houve) e segmentação (se viável).

In [ ]:
# Visualização detalhada de cada imagem do lote
for resultado in resultados:
    exibir_resultado(resultado)

---
## 8. Geração do Relatório CSV

O relatório consolida todas as métricas e decisões do pipeline em um CSV estruturado com 22 colunas:

| Grupo | Colunas |
|---|---|
| Identificação | arquivo, status_cliente |
| Decisão | needs_equalization, requires_human_review |
| Métricas originais | média, std, laplaciano, clipped black/white, issues |
| Métricas finais | média, std, laplaciano, clipped black/white, issues |
| Segmentação | separabilidade_otsu, limiar_otsu |
| Validação | correção tentada, aceita, ganho std, deslocamento média |

In [ ]:
# Gerar relatório CSV
csv_path = gerar_relatorio(resultados)
print(f"Relatório salvo em: {csv_path}")

---
## 9. Exibição e Download do Relatório

Visualização do CSV como tabela formatada com destaque por status.

In [ ]:
import pandas as pd

# Carregar e exibir o relatório
df = pd.read_csv(csv_path)

# Estilizar por status
def colorir_status(val):
    cores = {
        "ideal": "background-color: #2ecc71; color: white",
        "adequada": "background-color: #3498db; color: white",
        "corrigida_automaticamente": "background-color: #f39c12; color: white",
        "ajuste_sem_ganho_relevante": "background-color: #e67e22; color: white",
        "requer_revisao_humana": "background-color: #e74c3c; color: white",
    }
    return cores.get(val, "")

# Exibir tabela estilizada
styled = (
    df.style
    .map(colorir_status, subset=["status_cliente"])
    .format(precision=2, na_rep="—")
    .set_caption("Relatório Consolidado — EstetiScan AI")
)
display(styled)

In [ ]:
# Estatísticas resumidas
contagem = resumo["contagem"]
total = resumo["total"]

print("=" * 50)
print("  ESTATÍSTICAS DO PROCESSAMENTO")
print("=" * 50)
for status, count in contagem.items():
    pct = (count / total * 100) if total > 0 else 0
    barra = "█" * int(pct / 2.5)
    print(f"  {status:35s} {count:3d}  ({pct:5.1f}%)  {barra}")
print(f"  {'TOTAL':35s} {total:3d}")
print("=" * 50)

# Download do CSV no Colab
if IN_COLAB:
    files.download(str(csv_path))
    print(f"\nDownload iniciado: {csv_path.name}")
else:
    print(f"\nRelatório disponível em: {csv_path}")

---
## 10. Galeria de Amostras Visuais

Gera uma pasta organizada por categoria com exemplos do pipeline, ideal para:
- **Vídeo técnico** — selecionar casos representativos
- **Landing page** — mostrar antes/depois e casos complexos
- **Apresentação acadêmica** — evidenciar os 5 status do motor

A função `gerar_galeria()` produz um **painel-resumo PNG** e um **galeria.json** com os metadados.

In [ ]:
from motor.facade import gerar_galeria

# Gerar galeria de amostras visuais (até 3 exemplos por categoria)
galeria = gerar_galeria(resultados, max_por_categoria=3)

print(f"Pasta da galeria: {galeria['pasta']}")
print(f"Painel resumo:    {galeria['painel']}")
print(f"Resumo JSON:      {galeria['resumo_json']}")

In [ ]:
# Exibir o painel-resumo gerado
painel_path = galeria.get("painel")
if painel_path and Path(str(painel_path)).is_file():
    painel_img = cv2.imread(str(painel_path))
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(painel_img, cv2.COLOR_BGR2RGB))
    plt.title("Painel-Resumo — Amostras por Categoria", fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.show()
else:
    print("Painel não gerado (nenhuma amostra disponível).")

---

## Conclusão

O **EstetiScan AI** processou todas as imagens de forma 100% automática, sem ajuste manual por imagem.

O motor classificou cada imagem em uma das cinco categorias, gerando artefatos visuais e um relatório CSV consolidado.

**Destaques:**
- A análise de qualidade identifica problemas de brilho, contraste, nitidez e clipping
- A correção CLAHE é aplicada apenas quando necessária e validada antes de ser aceita
- A segmentação por Otsu só ocorre quando a separabilidade é suficiente (≥ 0.35)
- Imagens complexas são separadas para revisão humana automaticamente

> O sistema não substitui avaliação profissional. Atua como ferramenta de triagem, padronização e apoio visual, respeitando privacidade, consentimento e confidencialidade das imagens.